# Semantic Chunking + Dense/Sparse Embedding — BGE-M3 (Kaggle T4x2)

**Pipeline:** `tier_1.parquet` → Semantic Chunk (ColBERT) → Dense + Sparse Embed → Shard files

**Output artifacts:**
- `docs/` — metadata tài liệu (doc_id, url, host, lang, n_chunks)
- `chunks/` — metadata chunk (chunk_id, doc_id, start, end, n_chars, lang)
- `text/` — text store (doc_id, text gốc) để lấy nguyên văn
- `emb_dense/` — dense embeddings fp16 (1024 chiều)
- `emb_sparse/` — sparse lexical weights CSR

Tải artifacts về local → build FAISS + BM25 index tại local.

## 1. Cài đặt và Cấu hình (SỬA Ở ĐÂY)

In [1]:
!pip -q install pyarrow pandas bm25s jieba FlagEmbedding torch transformers

# ==================================================================
TEST_MODE = False  # <--- ĐỔI THÀNH False KHI MUỐN CHẠY FULL!
START_ROW = 400000       # Chạy từ dòng số bao nhiêu trong dataset
END_ROW   = 600000  # Dừng lại ở dòng số bao nhiêu (tăng dần 200000 mỗi session để tránh timeout)
# ==================================================================

## 2. Ẩn cảnh báo và import thư viện

In [ ]:
import os, json, math, time, glob, re, warnings

# Ẩn SyntaxWarning trước khi import jieba
warnings.filterwarnings('ignore', category=SyntaxWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='jieba')

from pathlib import Path
from typing import Iterable, Iterator, Tuple, List, Dict
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import torch
import jieba
from FlagEmbedding import BGEM3FlagModel

jieba.setLogLevel(60)

# ============================ CẤU HÌNH THEO MÔI TRƯỜNG ============================
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    print('Running on Kaggle')
    DATA_INPUT = '/kaggle/input'
    OUT_DIR = '/kaggle/working/index'
elif 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    print('Running on Colab')
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
    except ImportError:
        pass
    DATA_INPUT = '/content/drive/MyDrive/R2AI-Stage-3/data' # Thay đổi đường dẫn data theo Drive của bạn
    OUT_DIR = '/content/index'
else:
    print('Running Locally')
    DATA_INPUT = './data/raw'
    OUT_DIR = './data/indices/index'

IN_PARQUET    = sorted(glob.glob(f"{DATA_INPUT}/**/*.parquet", recursive=True))
MODEL_ID      = "BAAI/bge-m3"
FP16          = True

# Tăng batch size để tối ưu T4x2 (sẽ ngốn RAM hơn nhưng chạy nhanh hơn)
BATCH_SENTS   = 3072             # câu / lần forward (chunking)
BATCH_CHUNKS  = 1024              # chunk / lần forward (embedding)
MAX_LEN       = 8192             # bge-m3 tối đa 8192 token

# --- chunk semantic ---
SIM_THRESHOLD   = 0.75
MIN_INDEX_TOK   = 40
MAX_INDEX_TOK   = 400

# --- chunk để NỘP ---
SUBMISSION_TOK  = 1024
CHARS_PER_TOKEN = {"vi": 3.40, "zh": 1.30}

# --- lưu trữ ---
SHARD_CHUNKS   = 100_000
SHARD_DOCS     = 50_000
SHARD_TEXT     = 10_000
SAVE_SPARSE    = True

# --- lọc rác ---
# Đánh giá trên text đã gom whitespace, nhưng luôn chunk/embed text gốc để giữ offset.
DROP_HOSTS            = {}
MIN_QUALITY_CHARS     = 80
MIN_INFORMATIVE_CHARS = 50
MIN_UNIQUE_CHARS      = 15

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("IN:", IN_PARQUET)
print("OUT:", OUT_DIR)
print("TEST_MODE đang bật:", TEST_MODE)

IN: []
OUT: /kaggle/working/index
TEST_MODE đang bật: True


## 3. Đăng nhập HuggingFace (Kaggle Secrets)
Lấy `HF_TOKEN` từ Kaggle Secrets để tải mô hình nhanh và không bị rate-limit.

In [3]:
import os
from huggingface_hub import login

hf_token = None
try:
    # Thử lấy từ Kaggle Secrets
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    try:
        # Thử lấy từ Colab Secrets
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        # Lấy từ biến môi trường nếu chạy local
        hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("Đăng nhập HuggingFace thành công!")
else:
    print("Không tìm thấy HF_TOKEN trong Kaggle Secrets, Colab UserData, hoặc ENV variables.")


Không tìm thấy HF_TOKEN hoặc không chạy trên Kaggle: No module named 'kaggle_secrets'


## 4. Load model BGE-M3 (Sử dụng Multi-GPU)

In [4]:
num_gpus = torch.cuda.device_count()
gpu_list = [f"cuda:{i}" for i in range(num_gpus)] if num_gpus > 0 else None
model  = BGEM3FlagModel(MODEL_ID, use_fp16=FP16, devices=gpu_list)
print("model:", MODEL_ID, "| GPUs available:", num_gpus, "| devices:", gpu_list)

KeyboardInterrupt: 

## 5. Đọc corpus dạng stream

In [ ]:
def iter_docs(parquet_paths, batch_size=2000, start_row=0, end_row=None) -> Iterator[Tuple[int, str, str]]:
    current_row = 0
    for p in parquet_paths:
        pf = pq.ParquetFile(p)
        cols = [c for c in ("id", "url", "text") if c in pf.schema_arrow.names]
        for b in pf.iter_batches(batch_size=batch_size, columns=cols):
            batch_len = len(b)
            if current_row + batch_len <= start_row:
                current_row += batch_len
                continue
            
            d = b.to_pydict()
            ids = d.get("id") or [None]*batch_len
            urls = d.get("url") or [""]*batch_len
            texts = d.get("text", [])
            
            for i in range(batch_len):
                if current_row >= start_row:
                    if end_row is not None and current_row >= end_row:
                        return
                    doc_id = ids[i]
                    txt = texts[i]
                    if doc_id is not None and txt:
                        yield int(doc_id), (urls[i] or ""), txt
                current_row += 1

## 6. Lọc rác + nhận ngôn ngữ

Filter chỉ dùng bản chuẩn hoá whitespace để đánh giá. Text gốc vẫn được lưu và chunk verbatim, nhờ đó không làm lệch `start`/`end`. Ngưỡng cũ `len(text) >= 200` đã loại nhầm abstract và bài tiếng Trung ngắn có nội dung.

In [ ]:
CJK   = lambda c: "一" <= c <= "鿿"
HOST  = re.compile(r"https?://([^/:?#]+)")
SPACE = re.compile(r"\s+")
CHALLENGE_MARKERS = (
    "before accessing the website, it is necessary to verify the security of the connection and browser environment",
    "checking your browser before accessing",
    "enable javascript and cookies to continue",
    "verify you are human",
)

def host_of(u: str) -> str:
    m = HOST.match(u or "")
    h = (m.group(1) if m else "?").lower()
    return h[4:] if h.startswith("www.") else h

def quality_text(text: str) -> str:
    # Chỉ dùng cho gate/lang; tuyệt đối không thay text gốc dùng để tạo offset.
    return SPACE.sub(" ", text or "").strip()

def gate(url: str, text: str) -> Tuple[bool, str]:
    if host_of(url) in DROP_HOSTS:
        return False, "host chết"
    quality = quality_text(text)
    folded = quality.casefold()
    if any(marker in folded for marker in CHALLENGE_MARKERS):
        return False, "trang challenge"
    if len(quality) < MIN_QUALITY_CHARS:
        return False, "quá ngắn sau chuẩn hóa"
    informative = [c.casefold() for c in quality[:2000] if c.isalnum()]
    if len(informative) < MIN_INFORMATIVE_CHARS:
        return False, "thiếu nội dung chữ/số"
    if len(set(informative)) < MIN_UNIQUE_CHARS:
        return False, "độ đa dạng quá thấp"
    return True, ""

def lang_of(text: str) -> str:
    head = quality_text(text)[:500]
    letters = [c for c in head if c.isalpha()]
    return "zh" if sum(1 for c in letters if CJK(c)) / max(1, len(letters)) > 0.15 else "vi"


## 7. Tách câu

In [ ]:
SENT_ZH = re.compile(r"(?<=[。！？；!?;])")
SENT_VI = re.compile(r"(?<=[.!?;])\s+(?=[^\s])")
ABBREV = re.compile(r"(?:\b(?:BS|TS|ThS|GS|PGS|TSG|Tp|Q|T|H|P|X|ĐH|BV|TTYT|TLTK)\.)$", re.I)

def split_sentences(text: str, lang: str) -> List[Tuple[int, int]]:
    if lang == "zh":
        parts = SENT_ZH.split(text)
    else:
        parts = SENT_VI.split(text)
        merged, i = [], 0
        while i < len(parts):
            p = parts[i]
            while (i + 1 < len(parts) and ABBREV.search(p.rstrip())
                   and parts[i+1][:1].islower()):
                p = p + " " + parts[i+1]
                i += 1
            merged.append(p)
            i += 1
        parts = merged

    spans, pos = [], 0
    for p in parts:
        if not p: continue
        i = text.find(p, pos)
        if i < 0: continue
        j = i + len(p)
        if p.strip(): spans.append((i, j))
        pos = j
    return spans or [(0, len(text))]

## 8. Semantic chunking bằng ColBERT vectors

In [ ]:
def _norm(x):
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(n, 1e-12)

def _colbert_blocks(out, n: int) -> List[np.ndarray]:
    cv = out.get("colbert_vecs")
    if cv is None:
        return None
    lens = out.get("colbert_lens")
    if lens is None:
        return [np.asarray(v, dtype=np.float32) for v in cv]
    lens = np.asarray(lens).astype(np.int64).reshape(-1).tolist()
    if len(lens) != n:
        return [np.asarray(v, dtype=np.float32) for v in cv]
    cv = np.asarray(cv, dtype=np.float32)
    if cv.ndim == 2 and int(np.sum(lens)) == cv.shape[0]:
        b = np.cumsum([0] + lens)
        return [cv[b[i]:b[i+1]] for i in range(n)]
    return [np.asarray(v, dtype=np.float32) for v in cv]

@torch.no_grad()
def sentence_vectors(texts: List[str]) -> np.ndarray:
    V = []
    for i in range(0, len(texts), BATCH_SENTS):
        sub = texts[i:i+BATCH_SENTS]
        out = model.encode(sub, batch_size=BATCH_SENTS, max_length=512,
                           return_dense=True, return_sparse=False, return_colbert_vecs=True)
        blocks = _colbert_blocks(out, len(sub))
        if blocks is None:
            V.append(np.asarray(out["dense_vecs"], dtype=np.float32))
        else:
            V.append(np.stack([np.asarray(b, dtype=np.float32).mean(axis=0) for b in blocks]))
    return _norm(np.concatenate(V, axis=0))

def semantic_spans(text: str, lang: str) -> List[Tuple[int, int]]:
    cpt       = CHARS_PER_TOKEN.get(lang, 2.0)
    max_chars = int(MAX_INDEX_TOK * cpt)
    min_chars = int(MIN_INDEX_TOK * cpt)
    hard      = int(512 * cpt)

    spans = []
    for a, b in split_sentences(text, lang):
        while b - a > hard:
            spans.append((a, a + hard)); a += hard
        if b - a > 0:
            spans.append((a, b))
    if len(spans) <= 1:
        return spans or [(0, len(text))]

    sents = [text[a:b] for a, b in spans]
    V     = sentence_vectors(sents)

    groups, cur, cur_len = [], [0], len(sents[0])
    for i in range(1, len(sents)):
        sim = float(V[i] @ V[i-1])
        if sim >= SIM_THRESHOLD and cur_len + len(sents[i]) <= max_chars:
            cur.append(i); cur_len += len(sents[i])
        else:
            groups.append(cur); cur, cur_len = [i], len(sents[i])
    groups.append(cur)
    out = [(spans[g[0]][0], spans[g[-1]][1]) for g in groups]

    merged = []
    for a, b in out:
        if merged and (b - a < min_chars) and (b - merged[-1][0] <= max_chars):
            merged[-1] = (merged[-1][0], b)
        else:
            merged.append((a, b))
    return merged

## 9. Gộp thành submission chunk và Embed

In [ ]:
def merge_to_submission(spans, lang, budget_tok=SUBMISSION_TOK):
    limit = int(budget_tok * CHARS_PER_TOKEN.get(lang, 2.0))
    pieces = []
    for a, b in spans:
        while b - a > limit:
            pieces.append((a, a + limit)); a += limit
        if b - a > 0:
            pieces.append((a, b))
    out, cs, ce = [], None, None
    for a, b in pieces:
        if cs is None:
            cs, ce = a, b
        elif b - cs <= limit:
            ce = b
        else:
            out.append((cs, ce)); cs, ce = a, b
    if cs is not None:
        out.append((cs, ce))
    return out

@torch.no_grad()
def embed_chunks(texts: List[str]):
    out   = model.encode(texts, batch_size=BATCH_CHUNKS, max_length=MAX_LEN,
                         return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense  = np.asarray(out["dense_vecs"], dtype=np.float32)
    sparse = out["lexical_weights"]
    return dense, sparse

def sparse_to_arrays(weights: Dict[int, float]):
    if not weights:
        return np.zeros(0, np.int32), np.zeros(0, np.float32)
    ks = list(weights.keys())
    return np.asarray(ks, dtype=np.int32), np.asarray([weights[k] for k in ks], dtype=np.float32)

def sparse_to_csr(batch):
    ids, vals, ptr = [], [], [0]
    for w in batch:
        a, b = sparse_to_arrays(w)
        ids.append(a); vals.append(b); ptr.append(ptr[-1] + len(a))
    if not ids:
        return np.zeros(0, np.int32), np.zeros(0, np.float16), np.zeros(1, np.int64)
    return (np.concatenate(ids).astype(np.int32),
            np.concatenate(vals).astype(np.float16),
            np.asarray(ptr, dtype=np.int64))

## 10. Ghi ra đĩa — ShardedTable + ShardWriter

In [ ]:
class ShardedTable:
    def __init__(self, out_dir, columns, key="doc_id", shard_rows=100_000):
        self.dir = Path(out_dir); self.dir.mkdir(parents=True, exist_ok=True)
        self.cols = list(columns); self.key = key; self.shard_rows = shard_rows
        self._buf = []; self._parts = 0; self._idx = []
    def add(self, row):
        self._buf.append(row)
        if len(self._buf) >= self.shard_rows: self._flush()
    def _flush(self):
        if not self._buf: return
        df = pd.DataFrame(self._buf, columns=self.cols)
        pq.write_table(pa.Table.from_pandas(df, preserve_index=False),
                       self.dir / f"part_{self._parts:04d}.parquet")
        self._idx.append(np.asarray(df[self.key], dtype=np.int64))
        self._buf = []; self._parts += 1
    def close(self):
        self._flush()
        if self._idx:
            k = np.concatenate(self._idx)
            part = np.concatenate([np.full(len(x), i, np.int32) for i, x in enumerate(self._idx)])
            row = np.concatenate([np.arange(len(x), dtype=np.int32) for x in self._idx])
        else:
            k = part = row = np.zeros(0, np.int64)
        o = np.argsort(k, kind="stable")
        np.savez(self.dir / "_index.npz", key=k[o], part=part[o], row=row[o])
        return int(len(k))
    @classmethod
    def load(cls, out_dir, max_cached=2):
        self = cls.__new__(cls)
        z = np.load(Path(out_dir) / "_index.npz")
        self.dir = Path(out_dir)
        self._key = z["key"]; self._part = z["part"]; self._row = z["row"]
        self._cache = {}; self.max_cached = max_cached
        return self
    def __len__(self): return int(len(self._key))
    def nparts(self): return int(self._part.max()) + 1 if len(self._part) else 0
    def _shard(self, p):
        if p in self._cache: return self._cache[p]
        rec = pq.read_table(self.dir / f"part_{p:04d}.parquet").to_pylist()
        self._cache[p] = rec
        while len(self._cache) > self.max_cached: self._cache.pop(next(iter(self._cache)))
        return rec
    def row(self, k):
        i = int(np.searchsorted(self._key, int(k)))
        if i >= len(self._key) or int(self._key[i]) != int(k): raise KeyError(k)
        return self._shard(int(self._part[i]))[int(self._row[i])]

class ShardWriter:
    def __init__(self, out_dir, shard_size, prefix="vec"):
        self.dir = Path(out_dir); self.dir.mkdir(parents=True, exist_ok=True)
        self.shard_size = shard_size; self.prefix = prefix
        self._dv, self._si, self._sv, self._sp, self._ids = [], [], [], [], []
        self._n = 0; self._nid = 0; self.shards = []
    def add(self, dense=None, sids=None, svals=None, sptr=None, ids=None):
        n = len(dense) if dense is not None else len(ids) if ids is not None else (len(sptr) - 1) if sptr is not None else 0
        if n == 0: return
        if dense is not None: self._dv.append(np.asarray(dense, dtype=np.float16))
        if sids is not None:
            self._si.append(np.asarray(sids, dtype=np.int32))
            self._sv.append(np.asarray(svals, dtype=np.float16))
            self._sp.append(np.asarray(sptr, dtype=np.int64))
        if ids is None: ids = np.arange(self._nid, self._nid + n, dtype=np.int64)
        else: ids = np.asarray(ids, dtype=np.int64)
        self._ids.append(ids); self._nid += n; self._n += n
        while self._n >= self.shard_size: self.flush()
    def flush(self):
        if not self._n: return
        k = len(self.shards)
        if self._dv:
            np.save(self.dir / f"vec_{k:04d}.npy", np.concatenate(self._dv)); self._dv = []
        if self._ids:
            np.save(self.dir / f"ids_{k:04d}.npy", np.concatenate(self._ids)); self._ids = []
        if self._si:
            offs = np.cumsum([0] + [int(p[-1]) for p in self._sp]).astype(np.int64)
            ptr = np.concatenate([self._sp[i][:-1] + offs[i] for i in range(len(self._sp))] + [offs[-1:]]).astype(np.int64)
            np.savez(self.dir / f"sp_{k:04d}.npz", ids=np.concatenate(self._si), vals=np.concatenate(self._sv), ptr=ptr)
            self._si, self._sv, self._sp = [], [], []
        self.shards.append(k); self._n = 0
    def close(self):
        self.flush()
        return len(self.shards)

class TextStore:
    def __init__(self, path, max_cached=2):
        self.t = ShardedTable.load(path, max_cached=max_cached)
    def doc_text(self, doc_id) -> str:
        return self.t.row(doc_id)["text"]
    def chunk_text(self, chunk_row) -> str:
        t = self.doc_text(chunk_row["doc_id"])
        return t[int(chunk_row["start"]):int(chunk_row["end"])]
    def __len__(self): return len(self.t)

## 11. Vòng lặp chính & Kiểm tra

In [ ]:
def run_pass(parquet_paths, out_dir, limit=None, start_row=0, end_row=None):
    out_dir = Path(out_dir)
    dw     = ShardWriter(out_dir/"emb_dense", SHARD_CHUNKS)
    docs_t = ShardedTable(out_dir/"docs",   ["doc_id","url","host","lang","n_chunks"], "doc_id",   SHARD_DOCS)
    ch_t   = ShardedTable(out_dir/"chunks", ["chunk_id","doc_id","start","end","n_chars","lang"], "chunk_id", SHARD_DOCS)
    txt_t  = ShardedTable(out_dir/"text",   ["doc_id","text"], "doc_id", SHARD_TEXT)

    stats = {"docs":0, "kept":0, "chunks":0, "dropped":0, "drop_reason":{}, "failed":0}
    buf, buf_meta, next_cid, t0 = [], [], 0, time.time()

    def _flush_buf():
        nonlocal next_cid
        if not buf:
            return
        dense, sparse = embed_chunks(buf)
        sids, svals, sptr = sparse_to_csr(sparse) if SAVE_SPARSE else (None, None, None)
        ids = np.arange(next_cid, next_cid + len(buf), dtype=np.int64)
        if SAVE_SPARSE:
            dw.add(dense=dense, sids=sids, svals=svals, sptr=sptr, ids=ids)
        else:
            dw.add(dense=dense, ids=ids)
        for (doc_id, a, b, lang) in buf_meta:
            ch_t.add((next_cid, doc_id, a, b, b - a, lang))
            next_cid += 1
        stats["chunks"] += len(buf)
        buf.clear(); buf_meta.clear()

    for doc_id, url, txt in iter_docs(parquet_paths, start_row=start_row, end_row=end_row):
        stats["docs"] += 1
        if limit and stats["docs"] > limit:
            break
        ok, why = gate(url, txt)
        if not ok:
            stats["dropped"] += 1
            stats["drop_reason"][why] = stats["drop_reason"].get(why, 0) + 1
            continue
        lang = lang_of(txt)
        cpt  = CHARS_PER_TOKEN.get(lang, 2.0)
        try:
            if len(txt) > MAX_LEN * cpt:
                step = int(MAX_LEN * cpt)
                idx  = [(a, min(len(txt), a + step)) for a in range(0, len(txt), step)]
            else:
                idx  = semantic_spans(txt, lang)
            sub = merge_to_submission(idx, lang)
        except Exception:
            stats["failed"] += 1
            continue

        stats["kept"] += 1
        docs_t.add((doc_id, url, host_of(url), lang, len(sub)))
        txt_t.add((doc_id, txt))
        for a, b in sub:
            buf.append(txt[a:b])
            buf_meta.append((doc_id, a, b, lang))
        while len(buf) >= BATCH_CHUNKS:
            _flush_buf()
        if stats["docs"] % 5000 == 0:
            elapsed = time.time() - t0
            print(f"  {stats['docs']:,} docs | {stats['chunks']:,} chunk | {elapsed:.0f}s", flush=True)

    _flush_buf()
    dw.close()
    n_docs  = docs_t.close()
    n_chunk = ch_t.close()
    n_text  = txt_t.close()

    stats["rows_docs"], stats["rows_chunks"], stats["rows_text"] = n_docs, n_chunk, n_text
    stats["seconds"] = round(time.time() - t0, 1)
    assert n_chunk == next_cid == dw._nid, f"LỆCH: chunk {n_chunk} / next_cid {next_cid} / emb {dw._nid}"
    json.dump(stats, open(out_dir/"stats.json", "w"), ensure_ascii=False, indent=2)
    print(json.dumps(stats, ensure_ascii=False, indent=2))
    return stats

def verify(out_dir, n_sample=100, budget_tok=SUBMISSION_TOK):
    out_dir = Path(out_dir)
    ts = TextStore(out_dir/"text", max_cached=3)
    ch = ShardedTable.load(out_dir/"chunks", max_cached=1)
    rng = np.random.default_rng(0)
    N = len(ch)
    idxs = rng.choice(N, size=min(n_sample, N), replace=False) if N else []
    over = 0
    for i in idxs:
        r = ch.row(int(i))
        s = ts.chunk_text(r)
        assert s in ts.doc_text(r["doc_id"]), f"chunk {i} KHÔNG verbatim"
        ntok = len(model.tokenizer(s, add_special_tokens=False)["input_ids"])
        if ntok > budget_tok: over += 1
    print(f"[1] verbatim: OK ({len(idxs)} chunk) | vượt {budget_tok} token: {over}")
    for f in sorted(glob.glob(f"{out_dir}/emb_dense/sp_*.npz"))[:3]:
        z = np.load(f)
        assert z["ptr"][0] == 0 and z["ptr"][-1] == len(z["ids"]) == len(z["vals"])
        assert np.all(np.diff(z["ptr"]) >= 0)
        nchunk = len(z["ptr"]) - 1
        idsf = f.replace("/sp_", "/ids_").replace(".npz", ".npy")
        assert nchunk == len(np.load(idsf))
    print("[2] CSR ptr: OK")
    tot = 0
    for f in sorted(glob.glob(f"{out_dir}/emb_dense/ids_*.npy")):
        a = np.load(f)
        assert np.array_equal(a, np.arange(tot, tot + len(a)))
        tot += len(a)
    assert tot == len(ch)
    print(f"[3] chunk_id: OK ({tot:,} chunk)")
    for f in sorted(glob.glob(f"{out_dir}/emb_dense/vec_*.npy")):
        V = np.load(f, mmap_mode="r")
        assert V.dtype == np.float16 and V.ndim == 2
    print("[4] dense mmap: OK")
    return True

## 12. CHẠY PIPELINE (TỰ ĐỘNG CHỌN TEST HOẶC FULL DỰA VÀO CỜ TEST_MODE Ở ĐẦU)

In [ ]:
import shutil
shutil.rmtree(OUT_DIR, ignore_errors=True)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

if TEST_MODE:
    print("\n========== [TEST MODE] ĐANG CHẠY THỬ 200 TÀI LIỆU ==========\n")
    stats_test = run_pass(IN_PARQUET, OUT_DIR, limit=200, start_row=0, end_row=200)
    verify(OUT_DIR)
else:
    print(f"\n========== [FULL MODE] ĐANG CHẠY TỪ DÒNG {START_ROW} ĐẾN {END_ROW} ==========\n")
    stats = run_pass(IN_PARQUET, OUT_DIR, limit=None, start_row=START_ROW, end_row=END_ROW)
    verify(OUT_DIR)


## 13. Tổng kết

In [ ]:
total_bytes = 0
for root, dirs, files in os.walk(OUT_DIR):
    for f in files:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp)
        total_bytes += sz
print(f"\nTổng dung lượng artifacts: {total_bytes / 1e9:.2f} GB")

for d in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, d)
    if os.path.isdir(fp):
        n = len(os.listdir(fp))
        print(f"  {d}/  ({n} files)")
    else:
        print(f"  {d}  ({os.path.getsize(fp)/1e6:.1f} MB)")